# MILP v1 — Campus Microgrid Sizing (PuLP + HiGHS)

Two-stage stochastic MILP for optimal PV, BESS, and contract demand sizing.

- **Stage 1 (here-and-now):** PV capacity, BESS power/energy, contract demand
- **Stage 2 (recourse):** Hourly dispatch across 95 representative days × 5 scenarios
- **Solver:** HiGHS (open-source, no license required)

Uses shared config/data from `milp_common.py`.

In [ ]:
import numpy as np
import pulp
import time
from milp_common import get_config, load_data, print_results

In [ ]:
CFG = get_config()
scenarios, meta, repday_data, n_repdays, n_scenarios, n_hours = load_data(CFG)

## Build MILP Model

In [ ]:
prob = pulp.LpProblem('MicrogridSizing', pulp.LpMinimize)

# === Stage 1: Capacity variables ===
cap_pv = pulp.LpVariable('cap_pv', 0, CFG['pv_max_kw'])
cap_bess_p = pulp.LpVariable('cap_bess_p', 0, CFG['bess_p_max_kw'])
cap_bess_e = pulp.LpVariable('cap_bess_e', 0, CFG['bess_e_max_kwh'])
cap_contract = pulp.LpVariable('cap_contract', 0, None)

# === Stage 2: Dispatch variables ===
# Indexed by (day, scenario, hour)
p_grid = {}    # grid import
p_curt = {}    # PV curtailment
p_export = {}  # PV export (feed-in)
p_ch = {}      # BESS charge
p_dis = {}     # BESS discharge
soc = {}       # state of charge
over_contract = {}  # over-contract demand

for d in range(n_repdays):
    for s in range(n_scenarios):
        for t in range(n_hours):
            idx = (d, s, t)
            p_grid[idx] = pulp.LpVariable(f'grid_{d}_{s}_{t}', 0)
            p_curt[idx] = pulp.LpVariable(f'curt_{d}_{s}_{t}', 0)
            p_export[idx] = pulp.LpVariable(f'exp_{d}_{s}_{t}', 0)
            p_ch[idx] = pulp.LpVariable(f'ch_{d}_{s}_{t}', 0)
            p_dis[idx] = pulp.LpVariable(f'dis_{d}_{s}_{t}', 0)
            soc[idx] = pulp.LpVariable(f'soc_{d}_{s}_{t}', 0)
            over_contract[idx] = pulp.LpVariable(f'oc_{d}_{s}_{t}', 0)

print(f'Variables created: ~{7 * n_repdays * n_scenarios * n_hours + 4:,}')

## Objective & Constraints

In [ ]:
# === Objective: minimize annualized total cost ===
inv_cost = (
    cap_pv * CFG['capex_pv_per_kw'] * CFG['crf_pv']
    + cap_bess_p * CFG['capex_bess_power_per_kw'] * CFG['crf_bess']
    + cap_bess_e * (CFG['capex_bess_energy_per_kwh'] * CFG['crf_bess']
                    + CFG['capex_bess_energy_per_kwh'] * CFG['fom_bess_rate'])
    + cap_contract * CFG['contract_price_per_kw_month'] * 12
)

op_cost = pulp.lpSum(0)  # will accumulate

re_total = pulp.lpSum(0)   # RE numerator
load_total = pulp.lpSum(0) # RE denominator

PV_RATING = 50.0  # baseline PV rating in kW

for d in range(n_repdays):
    dd = repday_data[d]
    w = dd['weight']
    tou = dd['tou']

    for s in range(n_scenarios):
        sc = dd['scenarios'][s]
        pw = sc['prob'] * w  # probability × day-weight
        pv_avail = sc['pv_kw']
        load_kw = sc['load_kw']

        for t in range(n_hours):
            idx = (d, s, t)
            pv_coeff = float(pv_avail[t]) / PV_RATING

            # --- Power balance ---
            # pv_gen + grid + discharge = load + charge + curtailment + export
            prob += (
                pv_coeff * cap_pv + p_grid[idx] + p_dis[idx]
                == float(load_kw[t]) + p_ch[idx] + p_curt[idx] + p_export[idx]
            ), f'bal_{d}_{s}_{t}'

            # --- BESS limits ---
            prob += p_ch[idx] <= cap_bess_p, f'ch_lim_{d}_{s}_{t}'
            prob += p_dis[idx] <= cap_bess_p, f'dis_lim_{d}_{s}_{t}'

            # --- SOC dynamics ---
            if t == 0:
                soc_prev = CFG['soc_init'] * cap_bess_e
            else:
                soc_prev = soc[(d, s, t - 1)]

            prob += (
                soc[idx] == soc_prev
                + CFG['eff_charge'] * p_ch[idx]
                - (1.0 / CFG['eff_discharge']) * p_dis[idx]
            ), f'soc_{d}_{s}_{t}'

            prob += soc[idx] >= CFG['soc_min'] * cap_bess_e, f'soc_lo_{d}_{s}_{t}'
            prob += soc[idx] <= CFG['soc_max'] * cap_bess_e, f'soc_hi_{d}_{s}_{t}'

            # --- Over-contract penalty ---
            prob += over_contract[idx] >= p_grid[idx] - cap_contract, f'oc_{d}_{s}_{t}'

            # --- Export limit (no more than PV gen) ---
            prob += p_export[idx] <= pv_coeff * cap_pv, f'exp_lim_{d}_{s}_{t}'

            # --- Operating cost ---
            op_cost += pw * (
                p_grid[idx] * float(tou[t])
                + over_contract[idx] * CFG['penalty_per_kw']
                - p_export[idx] * CFG['feed_in_tariff']
                + (p_ch[idx] + p_dis[idx]) * CFG['bess_degradation_cost']
            )

            # --- RE accounting ---
            # RE = load served by non-grid = load - grid
            re_total += pw * (float(load_kw[t]) - p_grid[idx])
            load_total += pw * float(load_kw[t])

prob += inv_cost + op_cost, 'total_cost'

# === RE target constraint: sum(load - grid) >= target * sum(load) ===
prob += re_total >= CFG['re_target'] * load_total, 'RE30'

print(f'Constraints: {prob.numConstraints():,}')
print(f'Variables:   {prob.numVariables():,}')
print('Model built.')

## Solve

In [ ]:
solver = pulp.getSolver('HiGHS', msg=1, timeLimit=CFG['time_limit'])

t0 = time.time()
prob.solve(solver)
solve_time = time.time() - t0

print(f'\nStatus: {pulp.LpStatus[prob.status]}')
print(f'Objective: {pulp.value(prob.objective):,.0f} TWD')
print(f'Solve time: {solve_time:.1f}s')

## Results

In [ ]:
capacities = (
    pulp.value(cap_pv),
    pulp.value(cap_bess_p),
    pulp.value(cap_bess_e),
    pulp.value(cap_contract),
)
obj_val = pulp.value(prob.objective)
re_val = pulp.value(re_total)
load_val = pulp.value(load_total)

results = print_results(
    capacities, obj_val, CFG, re_val, load_val, solve_time,
    repday_data, n_repdays, n_scenarios, n_hours, scenarios
)